In [ ]:
### Geolibraries
import geopandas as gpd
import osmnx as ox
import contextily as ctx; import basemaps


# R5
import r5py
from r5py import TransportNetwork

# General tools
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
import pyarrow.parquet as pq

from h3 import h3
from shapely.geometry import Polygon

In [ ]:
users = pd.read_parquet("scratch/data/users_and_stays_3months_oulu.parquet")


In [ ]:
users_work = users[users["is_work"]==1]
users_work.to_parquet("./data/users_and_works_oulu.parquet")

In [ ]:
users = users[users["home_gid9"].notna()]

In [ ]:
users[users["is_home"]==1]

In [ ]:
stays_summary = (
    users
    .groupby("user_id")
    .agg(
        n_stays=("user_id", "count"),
        max_visit_frequency=("frequency_period", "max")
    )
    .reset_index()
)

In [ ]:
stays_summary.describe()

In [ ]:
plt.figure()
plt.hist(stays_summary["n_stays"], bins=30)
plt.xlabel("Number of stays per user")
plt.ylabel("Count")
plt.title("Distribution of number of stays per user")
plt.show()

In [ ]:
plt.figure()
plt.hist(stays_summary["max_visit_frequency"], bins=30)
plt.xlabel("Maximum visit frequency")
plt.ylabel("Count")
plt.title("Distribution of maximum visit frequency per user")
plt.show()

In [ ]:
plt.figure()
plt.hist(stays_summary["n_stays"], bins=100)
plt.yscale("log")
plt.xlabel("Number of stays per user")
plt.ylabel("Count (log scale)")
plt.title("Distribution of number of stays per user (log scale)")
plt.show()

In [ ]:
jobs = gpd.read_parquet("data/job_distribution_from_census_oulu.parquet")

In [ ]:
pois = pd.read_parquet("data/pois_per_hex_new_class_oulu.parquet")

In [ ]:
jobs = jobs.reset_index()

In [ ]:
users = users.merge(
    jobs[["ID", "weighted_tyo"]],
    left_on="stay_gid9",
    right_on="ID",
    how="left"
).drop(columns="ID")

In [ ]:
# Step 1: Group hsk_pois by 'h3_id' and 'category' to get counts
pois_grouped = (
    pois
    .groupby(['h3_id', 'category'])['count']
    .sum()                           # sum the values instead of counting rows
    .unstack(fill_value=0)           # make wide format, categories as columns
    .reset_index()
)


In [ ]:
pois_grouped

In [ ]:
users = users.merge(
    pois_grouped[
        [
            "h3_id",
            "Education",
            "Healthcare and Health",
            "Others / Not sure",
            "Recreational, Outdoors",
            "Shopping, Errands",
            "Social, Cultural",
        ]
    ],
    left_on="stay_gid9",
    right_on="h3_id",
    how="left"
).drop(columns="h3_id")

In [ ]:
users["user_id"].nunique()

In [ ]:
cols_needed = ["from_id", "to_id", "pt_co2_total"]
table_pt = pq.read_table("scratch/pt_co2_3000_oulu.parquet", 
                         columns=cols_needed,
                         use_threads=True)

df_pt_co2 = table_pt.to_pandas(types_mapper=pd.ArrowDtype)  # keeps pandas light

In [ ]:
df_pt_co2

In [ ]:
df_pt_co2_sym = df_pt_co2.merge(
    df_pt_co2,
    left_on=["from_id", "to_id"],
    right_on=["to_id", "from_id"],
    how="left",
    suffixes=("_outbound", "_inbound")
)

In [ ]:
df_pt_co2_sym

In [ ]:
df_pt_co2_sym["pt_co2_outbound"] = df_pt_co2_sym["pt_co2_total_outbound"]

df_pt_co2_sym["pt_co2_inbound"] = (
    df_pt_co2_sym["pt_co2_total_inbound"]
    .fillna(df_pt_co2_sym["pt_co2_total_outbound"])
)

df_pt_co2_sym["pt_co2_total"] = (
    df_pt_co2_sym["pt_co2_outbound"]
    + df_pt_co2_sym["pt_co2_inbound"]
)

In [ ]:
df_pt_co2_final = (
    df_pt_co2_sym
    .rename(columns={
        "from_id_outbound": "from_id",
        "to_id_outbound": "to_id"
    })[
        [
            "from_id",
            "to_id",
            "pt_co2_outbound",
            "pt_co2_inbound",
            "pt_co2_total"
        ]
    ]
)

In [ ]:
df_pt_co2 = df_pt_co2_final.copy()

In [ ]:
df_pt_co2

In [ ]:
users

In [ ]:
import pandas as pd

# ----------------------------
# STEP 1: Define activity columns
# ----------------------------

activity_cols = [
    "Social, Cultural",
    "Shopping, Errands",
    "Recreational, Outdoors"
]


# ----------------------------
# STEP 2: Filter non-home/work stays
# ----------------------------

df_base = users[
    (users["is_home"] == 0) &
    (users["is_work"] == 0)
].copy()


# ----------------------------
# STEP 3: Long format (KEEP METRICS!)
# ----------------------------

df_long = df_base.melt(
    id_vars=[
        "user_id",
        "stay_gid9",
        "home_gid9",
        "work_gid9",
        "frequency_period"   # IMPORTANT: preserve frequency
    ],
    value_vars=activity_cols,
    var_name="activity_type",
    value_name="poi_count"   # number of POIs of that type in stay
)

# Keep only active activity types
df_long = df_long[df_long["poi_count"] > 0].copy()

df_long = df_long.rename(columns={"stay_gid9": "activity_gid9"})



In [ ]:
df_long

In [ ]:

# ----------------------------
# STEP 4: Build tour-level rows (NOT legs yet)
# ----------------------------

df_long["tour_id"] = df_long.groupby(["user_id", "activity_type"]).cumcount()

tours = df_long.copy()



In [ ]:
tours

In [ ]:
# ----------------------------
# STEP 5: Expand into OD legs WITH METRICS ATTACHED
# ----------------------------

def build_legs(row):
    return [
        (row.user_id, row.home_gid9, row.work_gid9, row.activity_type,
         row.frequency_period, row.poi_count),

        (row.user_id, row.work_gid9, row.activity_gid9, row.activity_type,
         row.frequency_period, row.poi_count),

        (row.user_id, row.activity_gid9, row.home_gid9, row.activity_type,
         row.frequency_period, row.poi_count)
    ]

legs = tours.apply(build_legs, axis=1)

legs_df = pd.DataFrame(
    [leg for tour in legs for leg in tour],
    columns=[
        "user_id",
        "from_id",
        "to_id",
        "tour_type",
        "frequency",
        "poi_count"
    ]
)



In [ ]:
legs_df

In [ ]:
# ----------------------------
# STEP 6: Merge CO2
# ----------------------------

legs_df = legs_df.merge(
    df_pt_co2,
    on=["from_id", "to_id"],
    how="left"
)





In [ ]:
# ----------------------------
# STEP 7: Aggregate to full tour
# ----------------------------

legs_df["tour_id"] = range(len(legs_df))
legs_df["tour_id"] = legs_df["tour_id"] // 3


In [ ]:
tour_validity = (
    legs_df
    .groupby("tour_id")["pt_co2_outbound"]
    .count()
    .reset_index(name="n_legs")
)

In [ ]:
legs_df

In [ ]:
tour_validity

In [ ]:
valid_tours = tour_validity[
    tour_validity["n_legs"] == 3
]

In [ ]:
legs_df = legs_df.merge(
    valid_tours[["tour_id"]],
    on="tour_id",
    how="inner"
)

In [ ]:
legs_df

In [ ]:
tour_df = (
    legs_df
    .groupby(
        [
            "user_id",
            "tour_id",
            "tour_type",
            "frequency",
            "poi_count"
        ]
    )["pt_co2_outbound"]
    .sum()
    .reset_index()
    .rename(columns={"pt_co2_outbound": "tour_co2"})
)

In [ ]:
print("Total tours:", tour_validity.shape[0])
print("Valid tours:", (tour_validity["n_legs"] == 3).sum())
print("Discarded tours:", (tour_validity["n_legs"] < 3).sum())

In [ ]:
tour_df

In [ ]:
tour_df["exposure"] = (
    tour_df["frequency"] *
    tour_df["poi_count"]
)

In [ ]:
# At least 3 tours of that user-tour
tour_counts = (
    tour_df
    .groupby(["user_id", "tour_type"])
    .size()
    .reset_index(name="n_tours")
)

In [ ]:
valid_pairs = tour_counts[
    tour_counts["n_tours"] >= 3
]

In [ ]:
tour_df = tour_df.merge(
    valid_pairs[["user_id", "tour_type"]],
    on=["user_id", "tour_type"],
    how="inner"
)

In [ ]:
# Sort by CO2
tour_df = tour_df.sort_values(
    ["user_id", "tour_type", "tour_co2"]
)

tour_df["cum_exposure"] = (
    tour_df
    .groupby(["user_id", "tour_type"])["exposure"]
    .cumsum()
)

In [ ]:
# Normalize
tour_df["total_exposure"] = (
    tour_df
    .groupby(["user_id", "tour_type"])["exposure"]
    .transform("sum")
)

tour_df["cum_exposure_share"] = (
    tour_df["cum_exposure"] /
    tour_df["total_exposure"]
)

In [ ]:
import matplotlib.pyplot as plt

# =====================================
# USER INPUT
# =====================================

user_id = "00470a0e-dcaf-4203-ab27-7642131e06b5"

# =====================================
# FILTER USER
# =====================================

df_plot = tour_df[
    tour_df["user_id"] == user_id
].copy()

print("Categories found:")
print(df_plot["tour_type"].unique())

# =====================================
# PLOT
# =====================================

plt.figure(figsize=(10, 7))

for category in sorted(df_plot["tour_type"].unique()):

    df_cat = df_plot[
        df_plot["tour_type"] == category
    ].copy()

    df_cat = df_cat.sort_values("tour_co2")

    plt.step(
        df_cat["tour_co2"],
        df_cat["cum_exposure_share"],
        where="post",
        label=f"{category} (n={len(df_cat)})"
    )

    plt.scatter(
        df_cat["tour_co2"],
        df_cat["cum_exposure_share"],
        s=20
    )

plt.xlabel("Tour CO₂")
plt.ylabel("Cumulative Exposure Share")
plt.title(f"User {user_id}")

plt.xlim(left=0)
plt.ylim(0, 1.05)

plt.grid(True)
plt.legend()

plt.show()

In [ ]:
postal = gpd.read_file("./data/postal_code_finland/pno_tilasto_2024.shp")

In [ ]:
def h3_to_polygon(h3_id):
    return Polygon(h3.h3_to_geo_boundary(h3_id, geo_json=True))

users["geometry"] = users["home_gid9"].apply(h3_to_polygon)

In [ ]:
# Convert to GeoDataFrame
gdf = gpd.GeoDataFrame(users, geometry='geometry', crs="EPSG:4326")

In [ ]:
# Ensure both are in the same CRS
gdf = gdf.to_crs(postal.crs)

# Step 1: Compute intersections
intersections = gpd.overlay(gdf, postal, how='intersection')

# Step 2: Compute intersection area
intersections['intersect_area'] = intersections.geometry.area

In [ ]:
# Step 3: For each H3 hex, keep the postal code with the largest intersection
idx = intersections.groupby('home_gid9')['intersect_area'].idxmax()
largest_overlap = intersections.loc[idx]

# Step 4: Merge back the postal code to original H3 GeoDataFrame
gdf = gdf.merge(largest_overlap[['home_gid9', 'postinumer','nimi','geometry']], on='home_gid9', how='left')

In [ ]:
people_per_nimi = (
    gdf
    .groupby('nimi')
    .size()
    .reset_index(name='n_people')
)

In [ ]:
tour_df.columns

In [ ]:
tour_df = tour_df.merge(
    gdf[["user_id","home_gid9", "postinumer", "nimi"]].drop_duplicates(),
    on="user_id",
    how="left"
)

In [ ]:
tour_df

In [ ]:
df_typical = (
    tour_df
    .sort_values(
        ["user_id", "tour_type", "cum_exposure_share"]
    )
    .loc[tour_df["cum_exposure_share"] >= 0.5]
    .groupby(
        ["user_id", "tour_type", "nimi", "postinumer"],
        as_index=False
    )
    .first()
    .rename(columns={
        "tour_co2": "typical_trip_co2"
    })
)

In [ ]:
import matplotlib.pyplot as plt

df = df_typical.copy()

categories = df["tour_type"].unique()

fig, axes = plt.subplots(
    1,
    len(categories),
    figsize=(6 * len(categories), 5),
    sharey=True
)

if len(categories) == 1:
    axes = [axes]

for ax, cat in zip(axes, categories):

    df_cat = df[df["tour_type"] == cat]

    mean_val = df_cat["typical_trip_co2"].mean()
    median_val = df_cat["typical_trip_co2"].median()

    ax.hist(
        df_cat["typical_trip_co2"],
        bins=30,
        alpha=0.7,
        edgecolor="black"
    )

    ax.axvline(
        mean_val,
        color="red",
        linestyle="--",
        label=f"Mean: {mean_val:.2f}"
    )

    ax.axvline(
        median_val,
        color="blue",
        linestyle="-",
        label=f"Median: {median_val:.2f}"
    )

    ax.set_title(cat)
    ax.set_xlabel("Typical Trip CO₂")

    ax.legend()

axes[0].set_ylabel("Frequency")

plt.tight_layout()
plt.show()

In [ ]:
df_typical_single = pd.read_parquet("scratch/pt_typ_cat_oulu.parquet")

In [ ]:
df_typical_single

In [ ]:
df_typical

In [ ]:
df_final = df_typical.merge(
    df_typical_single,
    left_on=["user_id", "tour_type", "nimi", "postinumer"],
    right_on=["user_id", "poi_type", "nimi", "postinumer"],
    how="left",
    suffixes=("", "_single")
)

In [ ]:
df_final = df_final.drop(columns=["poi_type"], errors="ignore")

In [ ]:
df_jobs = df_typical_single[
    df_typical_single["poi_type"] == "jobs"
].copy()

df_jobs = df_jobs.rename(
    columns={"typical_trip_co2": "co2_jobs"}
)

In [ ]:
df_jobs

In [ ]:
df_jobs = df_jobs.rename(
    columns={"poi_type": "tour_type"}
)

df_jobs["tour_type"] = "jobs"

In [ ]:

df_jobs["tour_id"] = 0

df_jobs["frequency"] = 1
df_jobs["poi_count"] = 1



df_jobs["exposure"] = 1
df_jobs["cum_exposure"] = 1
df_jobs["total_exposure"] = 1
df_jobs["cum_exposure_share"] = 1



In [ ]:

df_jobs = df_jobs.rename(columns={"co2_jobs" : "typical_trip_co2"})

df_jobs["typical_trip_co2_single"] = df_jobs["typical_trip_co2"]

In [ ]:
df_jobs

In [ ]:
for col in df_final.columns:
    if col not in df_jobs.columns:
        df_jobs[col] = 1

In [ ]:
df_jobs = df_jobs[df_final.columns]

In [ ]:
df_final = pd.concat(
    [df_final, df_jobs],
    ignore_index=True
)

In [ ]:
df_final

In [ ]:
required_categories = {
    "jobs",
    "Social, Cultural",
    "Shopping, Errands",
    "Recreational, Outdoors"
}

# Users that have all required categories
valid_users = (
    df_final.groupby("user_id")["tour_type"]
    .apply(lambda x: required_categories.issubset(set(x)))
)

valid_users = valid_users[valid_users].index

df = df_final[df_final["user_id"].isin(valid_users)].copy()



worker_profile = {
    "jobs": 4,
    "Social, Cultural": 2,
    "Shopping, Errands": 1,
    "Recreational, Outdoors": 2
}

In [ ]:
tour = df.pivot(
    index="user_id",
    columns="tour_type",
    values="typical_trip_co2"
)

single = df.pivot(
    index="user_id",
    columns="tour_type",
    values="typical_trip_co2_single"
)

In [ ]:
weekly_routine = pd.DataFrame(index=tour.index)

weekly_routine["social_component"] = (
    2 * tour["Social, Cultural"]
)

weekly_routine["recreation_component"] = (
    2 * tour["Recreational, Outdoors"]
)

weekly_routine["shopping_component"] = (
    single["Shopping, Errands"]
)

weekly_routine["decent_mobility_co2"] = (
    weekly_routine["social_component"]
    + weekly_routine["recreation_component"]
    + weekly_routine["shopping_component"]
)

weekly_routine = weekly_routine.reset_index()

In [ ]:

# Budgets (grams CO2 / week)
BUDGET_2030 = 7000
BUDGET_2050 = 3000

threshold = 7000
total_users = len(weekly_routine)

pct_below_2030 = (weekly_routine["decent_mobility_co2"] <= BUDGET_2030).mean() * 100
pct_below_2050 = (weekly_routine["decent_mobility_co2"] <= BUDGET_2050).mean() * 100

fig, ax = plt.subplots(figsize=(10, 4))

# Boxplot
ax.boxplot(
    weekly_routine["decent_mobility_co2"].dropna(),
    vert=False,
    widths=0.5,
    patch_artist=True,
    boxprops=dict(facecolor="lightgray", edgecolor="black"),
    whiskerprops=dict(color="black"),
    capprops=dict(color="black"),
    medianprops=dict(color="black"),
    flierprops=dict(marker=".", markersize=3, alpha=0.4)
)

# Budget lines
ax.axvline(
    BUDGET_2030,
    color="tab:orange",
    linestyle="--",
    linewidth=2,
    label="2030 budget (7 kg CO₂ / week)"
)

ax.axvline(
    BUDGET_2050,
    color="tab:blue",
    linestyle="--",
    linewidth=2,
    label="2050 budget (3 kg CO₂ / week)"
)

# Annotations
ax.text(
    0.75, -0.10,
    f"{pct_below_2030:.1f}% below 2030 budget",
    color="tab:orange",
    fontsize=12,
    ha="left",
    va="top",
    transform=ax.transAxes
)

ax.text(
    0.75, -0.18,
    f"{pct_below_2050:.1f}% below 2050 budget",
    color="tab:blue",
    fontsize=12,
    ha="left",
    va="top",
    transform=ax.transAxes
)

# Formatting
ax.set_yticks([])
ax.set_xlabel("Weekly CO₂ emissions per worker (grams)", fontsize=11)
ax.set_title("Decent Mobility Routine – Weekly CO₂ Distribution", fontsize=14)

ax.legend(frameon=False)
ax.grid(axis="x", linestyle=":", alpha=0.5)

plt.tight_layout()
plt.show()

n_total = len(weekly_routine)
n_below = (weekly_routine["decent_mobility_co2"] < threshold).sum()
pct_below = 100 * n_below / n_total

print(f"{n_below:,} of {n_total:,} users ({pct_below:.1f}%) are below {threshold} g CO₂/week")

In [ ]:
weekly_routine

In [ ]:
home_df = (
    df[["user_id", "home_gid9"]]
    .dropna()
    .drop_duplicates(subset="user_id", keep="first")
)

In [ ]:
user_co2 = weekly_routine[["user_id", "decent_mobility_co2"]]

user_df = home_df.merge(
    user_co2,
    on="user_id",
    how="left"
)

In [ ]:
hex_df = (
    user_df
    .groupby("home_gid9", as_index=False)
    .agg(
        avg_co2=("decent_mobility_co2", "mean"),
        n_users=("user_id", "nunique")
    )
)

In [ ]:
# remove invalid / dummy H3 codes
hex_df = hex_df[
    hex_df["home_gid9"].notna() &
    (hex_df["home_gid9"] != 1) &
    (hex_df["home_gid9"] != "1")
]

In [ ]:
import h3
from shapely.geometry import Polygon

def h3_to_poly(h):
    boundary = h3.h3_to_geo_boundary(h, geo_json=True)
    return Polygon(boundary)

hex_df["geometry"] = hex_df["home_gid9"].apply(h3_to_poly)

gdf = gpd.GeoDataFrame(hex_df, geometry="geometry", crs="EPSG:4326")
gdf = gdf.to_crs(epsg=3857)

In [ ]:
import numpy as np
bins = [0, 1000, 3000, 7000, np.inf]
labels = ["0–1 kg", "1–3 kg", "3–7 kg", "7+ kg"]

gdf["co2_cat"] = pd.cut(
    gdf["avg_co2"],
    bins=bins,
    labels=labels,
    include_lowest=True
)

In [ ]:
filter_poly = gpd.read_file(
    "./data/oulu_filter_map.geojson"
).to_crs(epsg=3857)

In [ ]:
# NEW geopandas syntax
selection_geom = filter_poly.union_all()

In [ ]:
gdf_sel = gdf[gdf.within(selection_geom)]
#gdf_sel = gdf.copy()


colors = {
    "0–1 kg": "#08306b",   # very dark blue
    "1–3 kg": "#2171b5",   # strong medium blue
    "3–7 kg": "#9ecae1",   # clearly lighter (less saturated than before)
    "7+ kg": "#d73027"     # red (threshold violation)
}

fig, ax = plt.subplots(figsize=(10, 10))

for cat in labels:
    subset = gdf_sel[gdf_sel["co2_cat"] == cat]
    if len(subset) > 0:
        subset.plot(
            ax=ax,
            color=colors[cat],
            edgecolor="black",
            linewidth=0.2,
            alpha=0.9
        )

ctx.add_basemap(ax, source=basemaps.POSITRON)

ax.set_axis_off()
ax.set_title("Average Decent Mobility CO₂ per Home Hexagon", fontsize=14)

handles = [
    plt.Line2D([0], [0], marker='s',
               color=colors[l],
               linestyle='',
               markersize=10)
    for l in labels
]

ax.legend(handles, labels, title="Weekly CO₂ (kg)", loc="upper right")

plt.tight_layout()

plt.savefig(
    "./output/oulu_pt_decent_tour.png",
    dpi=300,
    bbox_inches="tight",
    facecolor="white"
)
plt.show()

In [ ]:
user_co2.to_parquet("./output/tour_pt_oulu_decent_per_user.parquet")

In [ ]:

df_final = df_final.copy()

# mark invalid values
df_final["home_gid9"] = df_final["home_gid9"].replace([1, "1"], np.nan)

# get first valid home per user
valid_home = (
    df_final
    .dropna(subset=["home_gid9"])
    .groupby("user_id")["home_gid9"]
    .first()
    .reset_index()
    .rename(columns={"home_gid9": "home_gid9_clean"})
)

df_final = df_final.merge(valid_home, on="user_id", how="left")

# overwrite bad/missing values
df_final["home_gid9"] = df_final["home_gid9_clean"]

df_final = df_final.drop(columns=["home_gid9_clean"])


df_final.to_parquet("./output/tour_pt_oulu_typical_per_user.parquet")

### Car

In [ ]:
import pyarrow.parquet as pq

cols_needed = [
    "Origin_Hexagon_ID","Destination_Hexagon_ID", "car_co2"
]

table = pq.read_table("scratch/car_co2_6000_oulu.parquet", 
                      columns=cols_needed,
                      use_threads=True)

df_car_co2 = table.to_pandas(types_mapper=pd.ArrowDtype)  # keeps pandas light

In [ ]:
df_car_co2 = df_car_co2.rename(
    columns={
        "Origin_Hexagon_ID": "from_id",
        "Destination_Hexagon_ID": "to_id"
    }
)

In [ ]:
df_car_co2_sym = df_car_co2.merge(
    df_car_co2,
    left_on=["from_id", "to_id"],
    right_on=["to_id", "from_id"],
    how="left",
    suffixes=("_outbound", "_inbound")
)

In [ ]:
df_car_co2_sym

In [ ]:
df_car_co2_sym["car_co2_outbound"] = df_car_co2_sym["car_co2_outbound"]

df_car_co2_sym["car_co2_inbound"] = (
    df_car_co2_sym["car_co2_inbound"]
    .fillna(df_car_co2_sym["car_co2_outbound"])
)

df_car_co2_sym["car_co2_total"] = (
    df_car_co2_sym["car_co2_outbound"]
    + df_car_co2_sym["car_co2_inbound"]
)

In [ ]:
df_car_co2_final = (
    df_car_co2_sym
    .rename(columns={
        "from_id_outbound": "from_id",
        "to_id_outbound": "to_id"
    })[
        [
            "from_id",
            "to_id",
            "car_co2_outbound",
            "car_co2_inbound",
            "car_co2_total"
        ]
    ]
)


In [ ]:
df_car_co2 = df_car_co2_final.copy()

In [ ]:
df_car_co2

In [ ]:

# ----------------------------
# STEP 1: Define activity columns
# ----------------------------
activity_cols = [
    "Social, Cultural",
    "Shopping, Errands",
    "Recreational, Outdoors"
]

# ----------------------------
# STEP 2: Filter non-home/work stays
# ----------------------------
df_base = users.loc[
    (users["is_home"] == 0) &
    (users["is_work"] == 0)
].copy()

# ----------------------------
# STEP 3: Long format (keep frequency!)
# ----------------------------
df_long = df_base.melt(
    id_vars=[
        "user_id",
        "stay_gid9",
        "home_gid9",
        "work_gid9",
        "frequency_period"
    ],
    value_vars=activity_cols,
    var_name="activity_type",
    value_name="poi_count"
)

# ----------------------------
# STEP 4: Keep only meaningful activity assignments
# ----------------------------
df_long = df_long[df_long["poi_count"] > 0].copy()

# ----------------------------
# STEP 5: Rename spatial column
# ----------------------------
df_long = df_long.rename(columns={"stay_gid9": "activity_gid9"})


In [ ]:
df_long

In [ ]:

# ----------------------------
# STEP 4: Build tour-level rows (NOT legs yet)
# ----------------------------

df_long["tour_id"] = df_long.groupby(["user_id", "activity_type"]).cumcount()

tours = df_long.copy()



In [ ]:
# ----------------------------
# STEP 5: Expand into OD legs WITH METRICS ATTACHED
# ----------------------------

def build_legs(row):
    return [
        (row.user_id, row.home_gid9, row.work_gid9, row.activity_type,
         row.frequency_period, row.poi_count),

        (row.user_id, row.work_gid9, row.activity_gid9, row.activity_type,
         row.frequency_period, row.poi_count),

        (row.user_id, row.activity_gid9, row.home_gid9, row.activity_type,
         row.frequency_period, row.poi_count)
    ]

legs = tours.apply(build_legs, axis=1)

legs_df = pd.DataFrame(
    [leg for tour in legs for leg in tour],
    columns=[
        "user_id",
        "from_id",
        "to_id",
        "tour_type",
        "frequency",
        "poi_count"
    ]
)




In [ ]:
# ----------------------------
# STEP 6: Merge CO2
# ----------------------------

legs_df = legs_df.merge(
    df_car_co2,
    on=["from_id", "to_id"],
    how="left"
)



In [ ]:
# ----------------------------
# STEP 7: Aggregate to full tour
# ----------------------------

legs_df["tour_id"] = range(len(legs_df))
legs_df["tour_id"] = legs_df["tour_id"] // 3


In [ ]:
tour_validity = (
    legs_df
    .groupby("tour_id")
    .agg(n_legs=("car_co2_outbound", "count"))
    .reset_index()
)

In [ ]:
tour_validity

In [ ]:
valid_tours = tour_validity[
    tour_validity["n_legs"] == 3
]

In [ ]:
legs_df = legs_df.merge(
    valid_tours[["tour_id"]],
    on="tour_id",
    how="inner"
)

In [ ]:
tour_df = (
    legs_df
    .groupby(
        [
            "user_id",
            "tour_id",
            "tour_type",
            "frequency",
            "poi_count"
        ]
    )["car_co2_outbound"]
    .sum()
    .reset_index()
    .rename(columns={"car_co2_outbound": "tour_co2"})
)

In [ ]:
print("Total tours:", tour_validity.shape[0])
print("Valid tours:", (tour_validity["n_legs"] == 3).sum())
print("Discarded tours:", (tour_validity["n_legs"] < 3).sum())

In [ ]:
tour_df

In [ ]:
tour_df["exposure"] = (
    tour_df["frequency"] *
    tour_df["poi_count"]
)

In [ ]:
# At least 3 tours of that user-tour
tour_counts = (
    tour_df
    .groupby(["user_id", "tour_type"])
    .size()
    .reset_index(name="n_tours")
)

In [ ]:
valid_pairs = tour_counts[
    tour_counts["n_tours"] >= 3
]

In [ ]:
tour_df = tour_df.merge(
    valid_pairs[["user_id", "tour_type"]],
    on=["user_id", "tour_type"],
    how="inner"
)

In [ ]:
# Sort by CO2
tour_df = tour_df.sort_values(
    ["user_id", "tour_type", "tour_co2"]
)

tour_df["cum_exposure"] = (
    tour_df
    .groupby(["user_id", "tour_type"])["exposure"]
    .cumsum()
)

In [ ]:
# Normalize
tour_df["total_exposure"] = (
    tour_df
    .groupby(["user_id", "tour_type"])["exposure"]
    .transform("sum")
)

tour_df["cum_exposure_share"] = (
    tour_df["cum_exposure"] /
    tour_df["total_exposure"]
)

In [ ]:
import matplotlib.pyplot as plt

# =====================================
# USER INPUT
# =====================================

user_id = "0003f83e-e647-4c6f-a9ab-3fd6cc294df8"

# =====================================
# FILTER USER
# =====================================

df_plot = tour_df[
    tour_df["user_id"] == user_id
].copy()

print("Categories found:")
print(df_plot["tour_type"].unique())

# =====================================
# PLOT
# =====================================

plt.figure(figsize=(10, 7))

for category in sorted(df_plot["tour_type"].unique()):

    df_cat = df_plot[
        df_plot["tour_type"] == category
    ].copy()

    df_cat = df_cat.sort_values("tour_co2")

    plt.step(
        df_cat["tour_co2"],
        df_cat["cum_exposure_share"],
        where="post",
        label=f"{category} (n={len(df_cat)})"
    )

    plt.scatter(
        df_cat["tour_co2"],
        df_cat["cum_exposure_share"],
        s=20
    )

plt.xlabel("Tour CO₂")
plt.ylabel("Cumulative Exposure Share")
plt.title(f"User {user_id}")

plt.xlim(left=0)
plt.ylim(0, 1.05)

plt.grid(True)
plt.legend()

plt.show()

In [ ]:
postal = gpd.read_file("./data/postal_code_finland/pno_tilasto_2024.shp")

In [ ]:
def h3_to_polygon(h3_id):
    return Polygon(h3.h3_to_geo_boundary(h3_id, geo_json=True))

users["geometry"] = users["home_gid9"].apply(h3_to_polygon)

In [ ]:
# Convert to GeoDataFrame
gdf = gpd.GeoDataFrame(users, geometry='geometry', crs="EPSG:4326")

In [ ]:
# Ensure both are in the same CRS
gdf = gdf.to_crs(postal.crs)

# Step 1: Compute intersections
intersections = gpd.overlay(gdf, postal, how='intersection')

# Step 2: Compute intersection area
intersections['intersect_area'] = intersections.geometry.area

In [ ]:
# Step 3: For each H3 hex, keep the postal code with the largest intersection
idx = intersections.groupby('home_gid9')['intersect_area'].idxmax()
largest_overlap = intersections.loc[idx]

# Step 4: Merge back the postal code to original H3 GeoDataFrame
gdf = gdf.merge(largest_overlap[['home_gid9', 'postinumer','nimi','geometry']], on='home_gid9', how='left')

In [ ]:
people_per_nimi = (
    gdf
    .groupby('nimi')
    .size()
    .reset_index(name='n_people')
)

In [ ]:
tour_df.columns

In [ ]:
tour_df = tour_df.merge(
    gdf[["user_id","home_gid9", "postinumer", "nimi"]].drop_duplicates(),
    on="user_id",
    how="left"
)

In [ ]:
tour_df

In [ ]:
df_typical = (
    tour_df
    .sort_values(
        ["user_id", "tour_type", "cum_exposure_share"]
    )
    .loc[tour_df["cum_exposure_share"] >= 0.5]
    .groupby(
        ["user_id", "tour_type", "nimi", "postinumer"],
        as_index=False
    )
    .first()
    .rename(columns={
        "tour_co2": "typical_trip_co2"
    })
)

In [ ]:
import matplotlib.pyplot as plt

df = df_typical.copy()

categories = df["tour_type"].unique()

fig, axes = plt.subplots(
    1,
    len(categories),
    figsize=(6 * len(categories), 5),
    sharey=True
)

if len(categories) == 1:
    axes = [axes]

for ax, cat in zip(axes, categories):

    df_cat = df[df["tour_type"] == cat]

    mean_val = df_cat["typical_trip_co2"].mean()
    median_val = df_cat["typical_trip_co2"].median()

    ax.hist(
        df_cat["typical_trip_co2"],
        bins=30,
        alpha=0.7,
        edgecolor="black"
    )

    ax.axvline(
        mean_val,
        color="red",
        linestyle="--",
        label=f"Mean: {mean_val:.2f}"
    )

    ax.axvline(
        median_val,
        color="blue",
        linestyle="-",
        label=f"Median: {median_val:.2f}"
    )

    ax.set_title(cat)
    ax.set_xlabel("Typical Trip CO₂")

    ax.legend()

axes[0].set_ylabel("Frequency")

plt.tight_layout()
plt.show()

In [ ]:
df_typical_single = pd.read_parquet("scratch/car_typ_cat_oulu.parquet")

In [ ]:
df_typical_single

In [ ]:
# Rename only if lowercase columns do not exist
rename_dict = {}

if "nimi" not in df_typical_single.columns and "Nimi" in df_typical_single.columns:
    rename_dict["Nimi"] = "nimi"

if "postinumer" not in df_typical_single.columns and "Posnro" in df_typical_single.columns:
    rename_dict["Posnro"] = "postinumer"

df_typical_single = df_typical_single.rename(columns=rename_dict)

In [ ]:
df_final = df_typical.merge(
    df_typical_single,
    left_on=["user_id", "tour_type", "nimi", "postinumer"],
    right_on=["user_id", "poi_type", "nimi", "postinumer"],
    how="left",
    suffixes=("", "_single")
)

In [ ]:
df_final = df_final.drop(columns=["poi_type"], errors="ignore")

In [ ]:
df_jobs = df_typical_single[
    df_typical_single["poi_type"] == "jobs"
].copy()

df_jobs = df_jobs.rename(
    columns={"typical_trip_co2": "co2_jobs"}
)

In [ ]:
df_jobs

In [ ]:
df_jobs = df_jobs.rename(
    columns={"poi_type": "tour_type"}
)

df_jobs["tour_type"] = "jobs"

In [ ]:

df_jobs["tour_id"] = 0

df_jobs["frequency"] = 1
df_jobs["poi_count"] = 1

df_jobs["exposure"] = 1
df_jobs["cum_exposure"] = 1
df_jobs["total_exposure"] = 1
df_jobs["cum_exposure_share"] = 1


In [ ]:

df_jobs = df_jobs.rename(columns={"co2_jobs" : "typical_trip_co2"})

df_jobs["typical_trip_co2_single"] = df_jobs["typical_trip_co2"]

In [ ]:
df_jobs

In [ ]:
for col in df_final.columns:
    if col not in df_jobs.columns:
        df_jobs[col] = 1

In [ ]:
df_jobs = df_jobs[df_final.columns]

In [ ]:
df_final = pd.concat(
    [df_final, df_jobs],
    ignore_index=True
)

In [ ]:
df_final

In [ ]:
required_categories = {
    "jobs",
    "Social, Cultural",
    "Shopping, Errands",
    "Recreational, Outdoors"
}

# Users that have all required categories
valid_users = (
    df_final.groupby("user_id")["tour_type"]
    .apply(lambda x: required_categories.issubset(set(x)))
)

valid_users = valid_users[valid_users].index

df = df_final[df_final["user_id"].isin(valid_users)].copy()



worker_profile = {
    "jobs": 4,
    "Social, Cultural": 2,
    "Shopping, Errands": 1,
    "Recreational, Outdoors": 2
}

In [ ]:
tour = df.pivot(
    index="user_id",
    columns="tour_type",
    values="typical_trip_co2"
)

single = df.pivot(
    index="user_id",
    columns="tour_type",
    values="typical_trip_co2_single"
)

In [ ]:
weekly_routine = pd.DataFrame(index=tour.index)

weekly_routine["social_component"] = (
    2 * tour["Social, Cultural"]
)

weekly_routine["recreation_component"] = (
    2 * tour["Recreational, Outdoors"]
)

weekly_routine["shopping_component"] = (
    single["Shopping, Errands"]
)

weekly_routine["decent_mobility_co2"] = (
    weekly_routine["social_component"]
    + weekly_routine["recreation_component"]
    + weekly_routine["shopping_component"]
)

weekly_routine = weekly_routine.reset_index()

In [ ]:

# Budgets (grams CO2 / week)
BUDGET_2030 = 7000
BUDGET_2050 = 3000

total_users = len(weekly_routine)

pct_below_2030 = (weekly_routine["decent_mobility_co2"] <= BUDGET_2030).mean() * 100
pct_below_2050 = (weekly_routine["decent_mobility_co2"] <= BUDGET_2050).mean() * 100

fig, ax = plt.subplots(figsize=(10, 4))

# Boxplot
ax.boxplot(
    weekly_routine["decent_mobility_co2"].dropna(),
    vert=False,
    widths=0.5,
    patch_artist=True,
    boxprops=dict(facecolor="lightgray", edgecolor="black"),
    whiskerprops=dict(color="black"),
    capprops=dict(color="black"),
    medianprops=dict(color="black"),
    flierprops=dict(marker=".", markersize=3, alpha=0.4)
)

# Budget lines
ax.axvline(
    BUDGET_2030,
    color="tab:orange",
    linestyle="--",
    linewidth=2,
    label="2030 budget (7 kg CO₂ / week)"
)

ax.axvline(
    BUDGET_2050,
    color="tab:blue",
    linestyle="--",
    linewidth=2,
    label="2050 budget (3 kg CO₂ / week)"
)

# Annotations
ax.text(
    0.75, -0.10,
    f"{pct_below_2030:.1f}% below 2030 budget",
    color="tab:orange",
    fontsize=12,
    ha="left",
    va="top",
    transform=ax.transAxes
)

ax.text(
    0.75, -0.18,
    f"{pct_below_2050:.1f}% below 2050 budget",
    color="tab:blue",
    fontsize=12,
    ha="left",
    va="top",
    transform=ax.transAxes
)

# Formatting
ax.set_yticks([])
ax.set_xlabel("Weekly CO₂ emissions per worker (grams)", fontsize=11)
ax.set_title("Decent Mobility Routine – Weekly CO₂ Distribution", fontsize=14)

ax.legend(frameon=False)
ax.grid(axis="x", linestyle=":", alpha=0.5)

plt.tight_layout()
plt.show()

n_total = len(weekly_routine)
n_below = (weekly_routine["decent_mobility_co2"] < BUDGET_2030).sum()
pct_below = 100 * n_below / n_total

print(f"{n_below:,} of {n_total:,} users ({pct_below:.1f}%) are below {BUDGET_2030} g CO₂/week")

In [ ]:
weekly_routine

In [ ]:
home_df = (
    df[["user_id", "home_gid9"]]
    .dropna()
    .drop_duplicates(subset="user_id", keep="first")
)

In [ ]:
user_co2 = weekly_routine[["user_id", "decent_mobility_co2"]]

user_df = home_df.merge(
    user_co2,
    on="user_id",
    how="left"
)

In [ ]:
hex_df = (
    user_df
    .groupby("home_gid9", as_index=False)
    .agg(
        avg_co2=("decent_mobility_co2", "mean"),
        n_users=("user_id", "nunique")
    )
)

In [ ]:
# remove invalid / dummy H3 codes
hex_df = hex_df[
    hex_df["home_gid9"].notna() &
    (hex_df["home_gid9"] != 1) &
    (hex_df["home_gid9"] != "1")
]

In [ ]:
import h3
from shapely.geometry import Polygon

def h3_to_poly(h):
    boundary = h3.h3_to_geo_boundary(h, geo_json=True)
    return Polygon(boundary)

hex_df["geometry"] = hex_df["home_gid9"].apply(h3_to_poly)

gdf = gpd.GeoDataFrame(hex_df, geometry="geometry", crs="EPSG:4326")
gdf = gdf.to_crs(epsg=3857)

In [ ]:
import numpy as np

bins = [0, 1000, 3000, 7000, np.inf]
labels = ["0–1 kg", "1–3 kg", "3–7 kg", "7+ kg"]

gdf["co2_cat"] = pd.cut(
    gdf["avg_co2"],
    bins=bins,
    labels=labels,
    include_lowest=True
)

In [ ]:
filter_poly = gpd.read_file(
    "./data/oulu_filter_map.geojson"
).to_crs(epsg=3857)

In [ ]:
# NEW geopandas syntax
selection_geom = filter_poly.union_all()

In [ ]:
gdf_sel = gdf[gdf.within(selection_geom)]
#gdf_sel = gdf.copy()


colors = {
    "0–1 kg": "#08306b",   # very dark blue
    "1–3 kg": "#2171b5",   # strong medium blue
    "3–7 kg": "#9ecae1",   # clearly lighter (less saturated than before)
    "7+ kg": "#d73027"     # red (threshold violation)
}

fig, ax = plt.subplots(figsize=(10, 10))

for cat in labels:
    subset = gdf_sel[gdf_sel["co2_cat"] == cat]
    if len(subset) > 0:
        subset.plot(
            ax=ax,
            color=colors[cat],
            edgecolor="black",
            linewidth=0.2,
            alpha=0.9
        )

ctx.add_basemap(ax, source=basemaps.POSITRON)

ax.set_axis_off()
ax.set_title("Average Decent Mobility CO₂ per Home Hexagon", fontsize=14)

handles = [
    plt.Line2D([0], [0], marker='s',
               color=colors[l],
               linestyle='',
               markersize=10)
    for l in labels
]

ax.legend(handles, labels, title="Weekly CO₂ (kg)", loc="upper right")

plt.tight_layout()

plt.savefig(
    "./output/oulu_car_decent_tour.png",
    dpi=300,
    bbox_inches="tight",
    facecolor="white"
)
plt.show()

In [ ]:
user_co2.to_parquet("./output/tour_car_oulu_decent_per_user.parquet")

In [ ]:

df_final = df_final.copy()

# mark invalid values
df_final["home_gid9"] = df_final["home_gid9"].replace([1, "1"], np.nan)

# get first valid home per user
valid_home = (
    df_final
    .dropna(subset=["home_gid9"])
    .groupby("user_id")["home_gid9"]
    .first()
    .reset_index()
    .rename(columns={"home_gid9": "home_gid9_clean"})
)

df_final = df_final.merge(valid_home, on="user_id", how="left")

# overwrite bad/missing values
df_final["home_gid9"] = df_final["home_gid9_clean"]

df_final = df_final.drop(columns=["home_gid9_clean"])


df_final.to_parquet("./output/tour_car_oulu_typical_per_user.parquet")

In [ ]:
df_final

### Bike

In [ ]:
import pyarrow.parquet as pq

cols_needed = [
    "from_id","to_id", "co2_emissions_g"
]

table = pq.read_table("scratch/bike_co2_3000_oulu.parquet", 
                      columns=cols_needed,
                      use_threads=True)

df_bike_co2 = table.to_pandas(types_mapper=pd.ArrowDtype)  # keeps pandas light


df_bike_co2 = df_bike_co2.rename(
    columns={"co2_emissions_g": "bike_co2"}
)

In [ ]:
outbound = df_bike_co2.rename(
    columns={"bike_co2": "bike_co2_outbound"}
)

inbound = (
    df_bike_co2
    .rename(
        columns={
            "from_id": "to_id",
            "to_id": "from_id",
            "bike_co2": "bike_co2_inbound"
        }
    )
)

df_bike_co2_sym = outbound.merge(
    inbound,
    on=["from_id", "to_id"],
    how="left"
)

df_bike_co2_sym["bike_co2_inbound"] = (
    df_bike_co2_sym["bike_co2_inbound"]
    .fillna(df_bike_co2_sym["bike_co2_outbound"])
)

df_bike_co2_sym["bike_co2_total"] = (
    df_bike_co2_sym["bike_co2_outbound"]
    + df_bike_co2_sym["bike_co2_inbound"]
)

In [ ]:
df_bike_co2_final = (
    df_bike_co2_sym[
        [
            "from_id",
            "to_id",
            "bike_co2_outbound",
            "bike_co2_inbound",
            "bike_co2_total"
        ]
    ]
)

In [ ]:
activity_cols = [
    "Social, Cultural",
    "Shopping, Errands",
    "Recreational, Outdoors"
]

df_long = (
    users.loc[
        (users["is_home"] == 0) &
        (users["is_work"] == 0)
    ]
    .melt(
        id_vars=[
            "user_id",
            "stay_gid9",
            "home_gid9",
            "work_gid9",
            "frequency_period"
        ],
        value_vars=activity_cols,
        var_name="activity_type",
        value_name="poi_count"
    )
)

df_long = (
    df_long[df_long["poi_count"] > 0]
    .rename(columns={"stay_gid9": "activity_gid9"})
)

In [ ]:

df_long["tour_id"] = df_long.groupby(["user_id", "activity_type"]).cumcount()

tours = df_long.copy()


def build_legs(row):
    return [
        (row.user_id, row.home_gid9, row.work_gid9, row.activity_type,
         row.frequency_period, row.poi_count),

        (row.user_id, row.work_gid9, row.activity_gid9, row.activity_type,
         row.frequency_period, row.poi_count),

        (row.user_id, row.activity_gid9, row.home_gid9, row.activity_type,
         row.frequency_period, row.poi_count)
    ]

legs = tours.apply(build_legs, axis=1)

legs_df = pd.DataFrame(
    [leg for tour in legs for leg in tour],
    columns=[
        "user_id",
        "from_id",
        "to_id",
        "tour_type",
        "frequency",
        "poi_count"
    ]
)


In [ ]:
df_bike_co2 = df_bike_co2_final.copy()

legs_df = legs_df.merge(
    df_bike_co2,
    on=["from_id", "to_id"],
    how="left"
)

legs_df["tour_id"] = range(len(legs_df))
legs_df["tour_id"] = legs_df["tour_id"] // 3

In [ ]:
legs_df

In [ ]:
tour_validity = (
    legs_df
    .groupby("tour_id")
    .agg(n_legs=("bike_co2_outbound", "count"))
    .reset_index()
)

tour_df = (
    legs_df
    .groupby(
        [
            "user_id",
            "tour_id",
            "tour_type",
            "frequency",
            "poi_count"
        ]
    )["bike_co2_outbound"]
    .sum()
    .reset_index()
    .rename(columns={"bike_co2_outbound": "tour_co2"})
)

In [ ]:
print("Total tours:", tour_validity.shape[0])
print("Valid tours:", (tour_validity["n_legs"] == 3).sum())
print("Discarded tours:", (tour_validity["n_legs"] < 3).sum())

In [ ]:
worker_profile = {
    "jobs": 4,
    "Social, Cultural": 2,
    "Shopping, Errands": 1,
    "Recreational, Outdoors": 2
}

In [ ]:
tour_df["exposure"] = (
    tour_df["frequency"] *
    tour_df["poi_count"]
)

tour_counts = (
    tour_df
    .groupby(["user_id", "tour_type"])
    .size()
    .reset_index(name="n_tours")
)

valid_pairs = tour_counts[
    tour_counts["n_tours"] >= 3
]


In [ ]:
tour_df = tour_df.merge(
    valid_pairs[["user_id", "tour_type"]],
    on=["user_id", "tour_type"],
    how="inner"
)

# Sort by CO2
tour_df = tour_df.sort_values(
    ["user_id", "tour_type", "tour_co2"]
)

tour_df["cum_exposure"] = (
    tour_df
    .groupby(["user_id", "tour_type"])["exposure"]
    .cumsum()
)

In [ ]:
# Normalize
tour_df["total_exposure"] = (
    tour_df
    .groupby(["user_id", "tour_type"])["exposure"]
    .transform("sum")
)

tour_df["cum_exposure_share"] = (
    tour_df["cum_exposure"] /
    tour_df["total_exposure"]
)

In [ ]:
# =====================================
# USER INPUT
# =====================================

user_id = "00020a53-96c5-4121-98e3-695cfa9e7b84"

# =====================================
# FILTER USER
# =====================================

df_plot = tour_df[
    tour_df["user_id"] == user_id
].copy()

print("Categories found:")
print(df_plot["tour_type"].unique())

# =====================================
# PLOT
# =====================================

plt.figure(figsize=(10, 7))

for category in sorted(df_plot["tour_type"].unique()):

    df_cat = df_plot[
        df_plot["tour_type"] == category
    ].copy()

    df_cat = df_cat.sort_values("tour_co2")

    plt.step(
        df_cat["tour_co2"],
        df_cat["cum_exposure_share"],
        where="post",
        label=f"{category} (n={len(df_cat)})"
    )

    plt.scatter(
        df_cat["tour_co2"],
        df_cat["cum_exposure_share"],
        s=20
    )

plt.xlabel("Tour CO₂")
plt.ylabel("Cumulative Exposure Share")
plt.title(f"User {user_id}")

plt.xlim(left=0)
plt.ylim(0, 1.05)

plt.grid(True)
plt.legend()

plt.show()

In [ ]:
postal = gpd.read_file("./data/postal_code_finland/pno_tilasto_2024.shp")

def h3_to_polygon(h3_id):
    return Polygon(h3.h3_to_geo_boundary(h3_id, geo_json=True))

users["geometry"] = users["home_gid9"].apply(h3_to_polygon)

# Convert to GeoDataFrame
gdf = gpd.GeoDataFrame(users, geometry='geometry', crs="EPSG:4326")


In [ ]:
# Ensure both are in the same CRS
gdf = gdf.to_crs(postal.crs)

# Step 1: Compute intersections
intersections = gpd.overlay(gdf, postal, how='intersection')

# Step 2: Compute intersection area
intersections['intersect_area'] = intersections.geometry.area

# Step 3: For each H3 hex, keep the postal code with the largest intersection
idx = intersections.groupby('home_gid9')['intersect_area'].idxmax()
largest_overlap = intersections.loc[idx]

# Step 4: Merge back the postal code to original H3 GeoDataFrame
gdf = gdf.merge(largest_overlap[['home_gid9', 'postinumer','nimi','geometry']], on='home_gid9', how='left')

In [ ]:
people_per_nimi = (
    gdf
    .groupby('nimi')
    .size()
    .reset_index(name='n_people')
)

In [ ]:
tour_df.columns

In [ ]:
tour_df = tour_df.merge(
    gdf[["user_id","home_gid9", "postinumer", "nimi"]].drop_duplicates(),
    on="user_id",
    how="left"
)

In [ ]:
df_typical = (
    tour_df
    .sort_values(
        ["user_id", "tour_type", "cum_exposure_share"]
    )
    .loc[tour_df["cum_exposure_share"] >= 0.5]
    .groupby(
        ["user_id", "tour_type", "nimi", "postinumer"],
        as_index=False
    )
    .first()
    .rename(columns={
        "tour_co2": "typical_trip_co2"
    })
)

In [ ]:
df = df_typical.copy()

categories = df["tour_type"].unique()

fig, axes = plt.subplots(
    1,
    len(categories),
    figsize=(6 * len(categories), 5),
    sharey=True
)

if len(categories) == 1:
    axes = [axes]

for ax, cat in zip(axes, categories):

    df_cat = df[df["tour_type"] == cat]

    mean_val = df_cat["typical_trip_co2"].mean()
    median_val = df_cat["typical_trip_co2"].median()

    ax.hist(
        df_cat["typical_trip_co2"],
        bins=30,
        alpha=0.7,
        edgecolor="black"
    )

    ax.axvline(
        mean_val,
        color="red",
        linestyle="--",
        label=f"Mean: {mean_val:.2f}"
    )

    ax.axvline(
        median_val,
        color="blue",
        linestyle="-",
        label=f"Median: {median_val:.2f}"
    )

    ax.set_title(cat)
    ax.set_xlabel("Typical Trip CO₂")

    ax.legend()

axes[0].set_ylabel("Frequency")

plt.tight_layout()
plt.show()

In [ ]:
df_typical_single = pd.read_parquet("scratch/bike_typ_cat_oulu.parquet")

In [ ]:
# Rename only if lowercase columns do not exist
rename_dict = {}

if "nimi" not in df_typical_single.columns and "Nimi" in df_typical_single.columns:
    rename_dict["Nimi"] = "nimi"

if "postinumer" not in df_typical_single.columns and "Posnro" in df_typical_single.columns:
    rename_dict["Posnro"] = "postinumer"

df_typical_single = df_typical_single.rename(columns=rename_dict)

In [ ]:

df_final = df_typical.merge(
    df_typical_single,
    left_on=["user_id", "tour_type", "nimi", "postinumer"],
    right_on=["user_id", "poi_type", "nimi", "postinumer"],
    how="left",
    suffixes=("", "_single")
)

In [ ]:
df_final = df_final.drop(columns=["poi_type"], errors="ignore")

In [ ]:
df_jobs = df_typical_single[
    df_typical_single["poi_type"] == "jobs"
].copy()

df_jobs = df_jobs.rename(
    columns={"typical_trip_co2": "co2_jobs"}
)

In [ ]:
df_jobs = df_jobs.rename(
    columns={"poi_type": "tour_type"}
)

df_jobs["tour_type"] = "jobs"

df_jobs["tour_id"] = 0

df_jobs["frequency"] = 1
df_jobs["poi_count"] = 1

df_jobs["exposure"] = 1
df_jobs["cum_exposure"] = 1
df_jobs["total_exposure"] = 1
df_jobs["cum_exposure_share"] = 1

In [ ]:

df_jobs = df_jobs.rename(columns={"co2_jobs" : "typical_trip_co2"})

df_jobs["typical_trip_co2_single"] = df_jobs["typical_trip_co2"]

In [ ]:
for col in df_final.columns:
    if col not in df_jobs.columns:
        df_jobs[col] = 1

df_jobs = df_jobs[df_final.columns]

df_final = pd.concat(
    [df_final, df_jobs],
    ignore_index=True
)

In [ ]:
required_categories = {
    "jobs",
    "Social, Cultural",
    "Shopping, Errands",
    "Recreational, Outdoors"
}

# Users that have all required categories
valid_users = (
    df_final.groupby("user_id")["tour_type"]
    .apply(lambda x: required_categories.issubset(set(x)))
)

valid_users = valid_users[valid_users].index

df = df_final[df_final["user_id"].isin(valid_users)].copy()



worker_profile = {
    "jobs": 4,
    "Social, Cultural": 2,
    "Shopping, Errands": 1,
    "Recreational, Outdoors": 2
}

In [ ]:
tour = df.pivot(
    index="user_id",
    columns="tour_type",
    values="typical_trip_co2"
)

single = df.pivot(
    index="user_id",
    columns="tour_type",
    values="typical_trip_co2_single"
)

In [ ]:
weekly_routine = pd.DataFrame(index=tour.index)

weekly_routine["social_component"] = (
    2 * tour["Social, Cultural"]
)

weekly_routine["recreation_component"] = (
    2 * tour["Recreational, Outdoors"]
)

weekly_routine["shopping_component"] = (
    single["Shopping, Errands"]
)

weekly_routine["decent_mobility_co2"] = (
    weekly_routine["social_component"]
    + weekly_routine["recreation_component"]
    + weekly_routine["shopping_component"]
)

weekly_routine = weekly_routine.reset_index()

In [ ]:
# Budgets (grams CO2 / week)
BUDGET_2030 = 7000
BUDGET_2050 = 3000

total_users = len(weekly_routine)

pct_below_2030 = (weekly_routine["decent_mobility_co2"] <= BUDGET_2030).mean() * 100
pct_below_2050 = (weekly_routine["decent_mobility_co2"] <= BUDGET_2050).mean() * 100

fig, ax = plt.subplots(figsize=(10, 4))

# Boxplot
ax.boxplot(
    weekly_routine["decent_mobility_co2"].dropna(),
    vert=False,
    widths=0.5,
    patch_artist=True,
    boxprops=dict(facecolor="lightgray", edgecolor="black"),
    whiskerprops=dict(color="black"),
    capprops=dict(color="black"),
    medianprops=dict(color="black"),
    flierprops=dict(marker=".", markersize=3, alpha=0.4)
)

# Budget lines
ax.axvline(
    BUDGET_2030,
    color="tab:orange",
    linestyle="--",
    linewidth=2,
    label="2030 budget (7 kg CO₂ / week)"
)

ax.axvline(
    BUDGET_2050,
    color="tab:blue",
    linestyle="--",
    linewidth=2,
    label="2050 budget (3 kg CO₂ / week)"
)

# Annotations
ax.text(
    0.75, -0.10,
    f"{pct_below_2030:.1f}% below 2030 budget",
    color="tab:orange",
    fontsize=12,
    ha="left",
    va="top",
    transform=ax.transAxes
)

ax.text(
    0.75, -0.18,
    f"{pct_below_2050:.1f}% below 2050 budget",
    color="tab:blue",
    fontsize=12,
    ha="left",
    va="top",
    transform=ax.transAxes
)

# Formatting
ax.set_yticks([])
ax.set_xlabel("Weekly CO₂ emissions per worker (grams)", fontsize=11)
ax.set_title("Decent Mobility Routine – Weekly CO₂ Distribution", fontsize=14)

ax.legend(frameon=False)
ax.grid(axis="x", linestyle=":", alpha=0.5)

plt.tight_layout()
plt.show()

n_total = len(weekly_routine)
n_below = (weekly_routine["decent_mobility_co2"] < BUDGET_2030).sum()
pct_below = 100 * n_below / n_total

print(f"{n_below:,} of {n_total:,} users ({pct_below:.1f}%) are below {BUDGET_2030} g CO₂/week")

In [ ]:
home_df = (
    df[["user_id", "home_gid9"]]
    .dropna()
    .drop_duplicates(subset="user_id", keep="first")
)

user_co2 = weekly_routine[["user_id", "decent_mobility_co2"]]

user_df = home_df.merge(
    user_co2,
    on="user_id",
    how="left"
)

hex_df = (
    user_df
    .groupby("home_gid9", as_index=False)
    .agg(
        avg_co2=("decent_mobility_co2", "mean"),
        n_users=("user_id", "nunique")
    )
)

In [ ]:
# remove invalid / dummy H3 codes
hex_df = hex_df[
    hex_df["home_gid9"].notna() &
    (hex_df["home_gid9"] != 1) &
    (hex_df["home_gid9"] != "1")
]

In [ ]:
filter_poly = gpd.read_file(
    "./data/oulu_filter_map.geojson"
).to_crs(epsg=3857)

# NEW geopandas syntax
selection_geom = filter_poly.union_all()

In [ ]:
import h3
from shapely.geometry import Polygon

def h3_to_poly(h):
    boundary = h3.h3_to_geo_boundary(h, geo_json=True)
    return Polygon(boundary)

hex_df["geometry"] = hex_df["home_gid9"].apply(h3_to_poly)

gdf = gpd.GeoDataFrame(hex_df, geometry="geometry", crs="EPSG:4326")
gdf = gdf.to_crs(epsg=3857)


bins = [0, 1000, 3000, 7000, np.inf]
labels = ["0–1 kg", "1–3 kg", "3–7 kg", "7+ kg"]

gdf["co2_cat"] = pd.cut(
    gdf["avg_co2"],
    bins=bins,
    labels=labels,
    include_lowest=True
)

In [ ]:
gdf_sel = gdf[gdf.within(selection_geom)]
#gdf_sel = gdf.copy()


colors = {
    "0–1 kg": "#08306b",   # very dark blue
    "1–3 kg": "#2171b5",   # strong medium blue
    "3–7 kg": "#9ecae1",   # clearly lighter (less saturated than before)
    "7+ kg": "#d73027"     # red (threshold violation)
}

fig, ax = plt.subplots(figsize=(10, 10))

for cat in labels:
    subset = gdf_sel[gdf_sel["co2_cat"] == cat]
    if len(subset) > 0:
        subset.plot(
            ax=ax,
            color=colors[cat],
            edgecolor="black",
            linewidth=0.2,
            alpha=0.9
        )

ctx.add_basemap(ax, source=basemaps.POSITRON)

ax.set_axis_off()
ax.set_title("Average Decent Mobility CO₂ per Home Hexagon", fontsize=14)

handles = [
    plt.Line2D([0], [0], marker='s',
               color=colors[l],
               linestyle='',
               markersize=10)
    for l in labels
]

ax.legend(handles, labels, title="Weekly CO₂ (kg)", loc="upper right")

plt.tight_layout()
plt.savefig(
    "./output/oulu_bike_decent_tour.png",
    dpi=300,
    bbox_inches="tight",
    facecolor="white"
)
plt.show()

In [ ]:
user_co2.to_parquet("./output/tour_bike_oulu_decent_per_user.parquet")

In [ ]:

df_final = df_final.copy()

# mark invalid values
df_final["home_gid9"] = df_final["home_gid9"].replace([1, "1"], np.nan)

# get first valid home per user
valid_home = (
    df_final
    .dropna(subset=["home_gid9"])
    .groupby("user_id")["home_gid9"]
    .first()
    .reset_index()
    .rename(columns={"home_gid9": "home_gid9_clean"})
)

df_final = df_final.merge(valid_home, on="user_id", how="left")

# overwrite bad/missing values
df_final["home_gid9"] = df_final["home_gid9_clean"]

df_final = df_final.drop(columns=["home_gid9_clean"])


df_final.to_parquet("./output/tour_bike_oulu_typical_per_user.parquet")

In [ ]:
df_final